In [ ]:
import ssl
import certifi
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import urllib.request
import io

chemin_titanic = Path("../exercice-08/data/titanic_propre.csv")
if not chemin_titanic.exists():
    chemin_titanic = Path("exercice-08/data/titanic_propre.csv")
titanic = pd.read_csv(chemin_titanic)

url_taxis = "https://raw.githubusercontent.com/seaborn-data/seaborn-data/master/taxis.csv"
context = ssl.create_default_context(cafile=certifi.where())
with urllib.request.urlopen(url_taxis, context=context) as response:
    taxis = pd.read_csv(io.BytesIO(response.read()))

titanic["age"] = pd.to_numeric(titanic["age"], errors="coerce")
titanic["fare"] = pd.to_numeric(titanic["fare"], errors="coerce")
taxis["pickup"] = pd.to_datetime(taxis["pickup"], errors="coerce")
for colonne in ["distance", "fare", "total"]:
    taxis[colonne] = pd.to_numeric(taxis[colonne], errors="coerce")

dossier_exercice = Path("exercice-12") if Path("exercice-12").is_dir() else Path(".")
dossier_figures = dossier_exercice / "figures"
dossier_figures.mkdir(parents=True, exist_ok=True)
print("Données chargées ; graphiques enregistrés dans figures/.")

## 1. Quel âge avaient les passagers ?

Un histogramme convient pour montrer la distribution d’une variable quantitative comme l’âge.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=titanic, x="age", bins=30, ax=ax)
ax.set(title="La plupart des passagers avaient entre 20 et 40 ans", xlabel="Âge (années)", ylabel="Nombre de passagers")
fig.savefig(dossier_figures / "01_ages.png", dpi=150, bbox_inches="tight")
plt.show()

La distribution est concentrée chez les jeunes adultes, surtout entre 20 et 40 ans. Les âges manquants ne figurent pas dans l’histogramme.

## 2. La survie selon la classe et le sexe

Un diagramme en barres compare les taux de survie entre catégories de classe et de sexe.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=titanic, x="pclass", y="survived", hue="sex", errorbar=None, ax=ax)
ax.set(title="Les femmes ont davantage survécu dans chaque classe", xlabel="Classe (1 = première)", ylabel="Taux de survie")
ax.set_ylim(0, 1)
fig.savefig(dossier_figures / "02_survie_classe_sexe.png", dpi=150, bbox_inches="tight")
plt.show()

Le taux de survie est plus élevé chez les femmes dans chacune des classes ; les passagers de première classe ont globalement les meilleurs résultats.

## 3. Le prix du billet selon la classe

Un boxplot compare la distribution des tarifs par classe et rend visibles les valeurs atypiques.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=titanic, x="pclass", y="fare", ax=ax)
ax.set(title="Les billets de première classe coûtaient le plus cher", xlabel="Classe (1 = première)", ylabel="Prix du billet (livres sterling)")
fig.savefig(dossier_figures / "03_prix_classe.png", dpi=150, bbox_inches="tight")
plt.show()

La médiane du prix augmente nettement avec le niveau de classe. Quelques tarifs très élevés restent visibles comme valeurs atypiques.

## 4. Chiffre d’affaires des taxis jour après jour

Une courbe montre l’évolution du chiffre d’affaires au fil du temps.

In [ ]:
ca_par_jour = taxis.dropna(subset=["pickup"]).set_index("pickup")["total"].resample("D").sum()
fig, ax = plt.subplots(figsize=(11, 5))
ca_par_jour.plot(ax=ax)
ax.set(title="Le chiffre d’affaires quotidien varie d’un jour à l’autre", xlabel="Date", ylabel="Chiffre d’affaires ($)")
fig.autofmt_xdate()
fig.savefig(dossier_figures / "04_ca_quotidien_taxis.png", dpi=150, bbox_inches="tight")
plt.show()

Le chiffre d’affaires quotidien fluctue fortement ; les journées avec peu de courses apparaissent comme des creux. Les sommes sont calculées sur les courses dont la date de prise en charge est valide.

## 5. Lien entre distance et prix des taxis

Un nuage de points permet d’observer le lien entre deux variables quantitatives.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=taxis, x="distance", y="fare", alpha=0.35, ax=ax)
ax.set(title="Le prix de la course augmente généralement avec la distance", xlabel="Distance (miles)", ylabel="Tarif ($)")
fig.savefig(dossier_figures / "05_distance_tarif.png", dpi=150, bbox_inches="tight")
plt.show()

Le nuage montre une association positive : les courses plus longues coûtent généralement davantage, avec une dispersion liée aux autres facteurs de tarification.